# Quantum Vision Transformer (QViT) vs. Classical ViT
## University Sem 5 Quantum Computing & Algorithms Course Project
**Student Research Study**: Medical Image Classification (Chest X-Ray Pneumonia Kermany Benchmark)  
**Frameworks**: PyTorch & PennyLane Quantum Simulator (`default.qubit` & `default.mixed`)  

---
### Abstract
This study compares a standard **Classical Vision Transformer (ViT)** against a hybrid **Quantum Vision Transformer (QViT)**. In the QViT, classical dot-product self-attention $\text{softmax}(QK^T / \sqrt{d})V$ is replaced by **Quantum Self-Attention (QSA)**, where the attention weights between image patches are determined by calculating quantum state fidelity $|\langle \psi_q | \phi_k \rangle|^2$ using an Angle-Embedded **Quantum SWAP Test Circuit**.

In [ ]:
import os
import json
import torch
import pennylane as qml
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

from src.models.classical_vit import ClassicalViT
from src.models.qvit import QViT
from src.data.dataset import load_medical_dataset
from src.quantum.swap_test import QuantumSwapTestModule, test_swap_test_properties
from src.utils.visualization import set_plot_style, plot_training_curves, plot_attention_heatmaps, plot_noise_resilience, format_parameter_table

print(f"PyTorch Version: {torch.__version__}")
print(f"PennyLane Version: {qml.__version__}")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

## 1. Quantum SWAP Test Circuit Verification
The SWAP test computes the inner product between two quantum states $|\psi_q\rangle$ and $|\phi_k\rangle$:
$$\langle Z_a \rangle = P(|0\rangle_a) - P(|1\rangle_a) = |\langle \psi_q | \phi_k \rangle|^2 = \text{Fidelity} \in [0, 1]$$

Let's run the mathematical verification module to confirm:
1. Identical states yield fidelity $= 1.0000$
2. PyTorch gradients propagate back through the quantum circuit
3. Batch parallel evaluations run efficiently

In [ ]:
test_swap_test_properties()

## 2. Authentic Model Parameter Count Comparison
We instantiate both models with identical configurations ($128 \times 128$ images, $32 \times 32$ patches, embedding dimension 32, 2 encoder layers) and compute parameter counts directly from `model.parameters()`.

In [ ]:
classical_model = ClassicalViT(image_size=128, patch_size=32, embed_dim=32, depth=2, num_heads=2, num_classes=2)
qvit_model = QViT(image_size=128, patch_size=32, embed_dim=32, n_qubits=4, depth=2, num_classes=2)

table_text = format_parameter_table(classical_model, qvit_model)
print(table_text)

## 3. Training & Validation Overlaid Curves
Here we load the recorded training metrics for both models and plot the overlaid loss and accuracy trajectories.

In [ ]:
classical_metrics_file = "./results/classical_vit_metrics.json"
qvit_metrics_file = "./results/qvit_metrics.json"

if os.path.exists(classical_metrics_file) and os.path.exists(qvit_metrics_file):
    with open(classical_metrics_file) as f:
        c_metrics = json.load(f)
    with open(qvit_metrics_file) as f:
        q_metrics = json.load(f)
    
    plot_training_curves(c_metrics, q_metrics, save_path="./results/training_curves.png")
    
    # Display plot directly in notebook
    from IPython.display import Image, display
    display(Image(filename="./results/training_curves.png"))
else:
    print("Metrics files not found yet. Run training scripts first.")

## 4. Quantum Attention Heatmaps on Medical Test Images
We pass sample test images through the trained QViT, extract the learned quantum attention matrix between the `[CLS]` token and all image patches, and overlay the heatmap on the medical image using alpha blending.

In [ ]:
heatmaps_file = "./results/attention_heatmaps.png"
if os.path.exists(heatmaps_file):
    from IPython.display import Image, display
    display(Image(filename=heatmaps_file))
else:
    print("Attention heatmaps image not generated yet. Run evaluate_and_analyze.py.")

## 5. NISQ Noise Resilience Analysis
To assess how real-world quantum hardware noise affects model performance, we evaluate the trained QViT under PennyLane's `default.mixed` simulator with depolarizing noise rates $p \in [0.0, 0.20]$.

In [ ]:
noise_file = "./results/noise_degradation_curve.png"
if os.path.exists(noise_file):
    from IPython.display import Image, display
    display(Image(filename=noise_file))
else:
    print("Noise degradation curve not generated yet. Run evaluate_and_analyze.py.")

## 6. Key Conclusions & Viva Defense Summary
1. **Parameter Efficiency**: QViT achieves an authentic reduction in trainable parameters because Query ($Q$) and Key ($K$) projections map directly down to 4 qubits per register, replacing $d_{\text{embed}} \times d_{\text{embed}}$ projection matrices.
2. **Attention Expressivity**: The quantum state fidelity $|\langle \psi_q | \phi_k \rangle|^2$ provides a valid, bounded similarity metric in $[0, 1]$.
3. **Simulation Trade-Off**: Pairwise SWAP tests over $N$ patches require $O(N^2)$ circuit evaluations. On a classical CPU simulator, this introduces computational overhead compared to BLAS matrix multiplications, but on native quantum hardware, state overlap runs in $O(1)$ depth.
4. **Noise Vulnerability**: Depolarizing noise degrades classification accuracy as quantum coherence is lost, emphasizing the need for quantum error mitigation.